# Video-LLaVA / MuLER Stage 1 Pretraining on Google Colab (Drive-First)

### Architecture & Pipeline Overview
Since Colab's ephemeral SSD (~80 GB) cannot hold the full datasets (images: 27 GB, videos: 460+ GB),
we use **Google Drive as the primary data store**:

| Component | Location | Why |
|-----------|----------|-----|
| Image dataset (558K images) | Google Drive | Too large for SSD |
| Video dataset (Valley) | Google Drive | Way too large for SSD |
| Annotation JSONs | Local SSD (copied from Drive) | Tiny files, fast reads |
| Model cache | Local SSD | Fast loading |
| Training checkpoints | Local SSD → auto-synced to Drive | Fast writes, persistent backups |

### Datasets Used:
- **Image Pretrain**: LLaVA-558K (`llava_image.zip` from HuggingFace)
- **Video Pretrain**: Valley (`valley_2.zip.001`-`012` from HuggingFace)
- **Annotations**: Official zip from Video-LLaVA authors

## 1. Check GPU Environment

In [ ]:
!nvidia-smi

## 2. Mount Google Drive

> **Partner Setup Note (Shared Folder)**:
> If accessing the dataset from a shared Google Drive folder:
> 1. Go to Google Drive web (`drive.google.com`) -> **Shared with me**.
> 2. Right-click the **Video-LLaVA** folder -> **Add shortcut to Drive** -> select **My Drive**.
> 3. Run the cell below to mount and verify.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = "/content/drive/MyDrive/Video-LLaVA"
os.makedirs(f"{DRIVE_ROOT}/datasets", exist_ok=True)
os.makedirs(f"{DRIVE_ROOT}/checkpoints", exist_ok=True)
print(f"Google Drive workspace ready at: {DRIVE_ROOT}")

# Verification Check
valley_path = f"{DRIVE_ROOT}/datasets/valley"
image_path = f"{DRIVE_ROOT}/datasets/llava_image"
if os.path.exists(valley_path) and os.path.exists(image_path):
    print("\u2705 Datasets verified on Google Drive! Ready for pretraining.")
else:
    print("\u26a0\ufe0f Note: Datasets not found at default path. If using a shared folder, ensure 'Add shortcut to Drive' was performed in MyDrive.")

## 3. Clone Repository & Install Dependencies
Installs PyTorch dependencies, DeepSpeed ZeRO-2, and optional Flash Attention for A100 acceleration.

In [ ]:
%cd /content

# Clone the repo (skip if already cloned)
![ ! -d "grad_project" ] && git clone https://github.com/davidrimon2004/grad_project
%cd /content/grad_project

# Pull latest updates
!git pull

# Install dependencies (uses Colab's pre-installed PyTorch)
!pip install -q --upgrade pip
!pip install -q transformers tokenizers sentencepiece shortuuid accelerate peft bitsandbytes einops einops-exts timm deepspeed huggingface_hub decord gdown av
# Flash Attention accelerates attention computation by 2-3x on A100 (Ampere)
!pip install -q flash-attn --no-build-isolation || echo "Flash attention installation skipped; fallback to standard PyTorch attention."
!apt-get install -y -qq p7zip-full
!pip install -e .

## 4. Option A: Quick Demo / Verification (100 Samples)
Runs a fast dry run on 100 synthetic samples on local SSD in ~2 minutes to verify GPU, forward/backward pass, and DeepSpeed before full training.

In [ ]:
!python scripts/colab_pretrain_muler.py \
    --action all \
    --demo_samples 100 \
    --drive_root /content/drive/MyDrive/Video-LLaVA \
    --local_scratch_dir /content/data \
    --local_output_dir /content/checkpoints/videollava-7b-demo \
    --num_train_epochs 1.0 \
    --save_steps 25

## 5. Verify Dataset & System Status
Checks disk space and verifies that images (558K) and Valley videos (228K+) are detected on Drive.

In [ ]:
!python scripts/colab_pretrain_muler.py \
    --action status \
    --drive_root /content/drive/MyDrive/Video-LLaVA \
    --local_scratch_dir /content/data

## 6. Pretraining Stage 1 (A100 Optimized)

Both setups below read images & videos directly from Google Drive, automatically detect the A100 GPU (enabling native **`bf16`**, **`tf32`**, and micro-batch size 8 $\times$ 4 gradient accumulation = effective batch size 32), and auto-save checkpoints to Drive every 500 steps.

### 6A. Proposal Pretraining (MuLER)
Trains the proposed MuLER architecture and isolates weights into `checkpoints/muler-7b-pretrain/`.

In [ ]:
!python scripts/colab_pretrain_muler.py \
    --action train \
    --drive_root /content/drive/MyDrive/Video-LLaVA \
    --local_scratch_dir /content/data \
    --local_output_dir /content/checkpoints/muler-7b-pretrain \
    --learning_rate 1e-3 \
    --num_train_epochs 1.0 \
    --save_steps 500 \
    --save_total_limit 2

### 6B. Baseline Pretraining (Optional)
Trains the standard Video-LLaVA baseline from scratch (only needed if an exact local comparison is required). Saves to `checkpoints/videollava-7b-pretrain/`.

In [ ]:
!python scripts/colab_pretrain_muler.py \
    --action train \
    --drive_root /content/drive/MyDrive/Video-LLaVA \
    --local_scratch_dir /content/data \
    --local_output_dir /content/checkpoints/videollava-7b-pretrain \
    --learning_rate 1e-3 \
    --num_train_epochs 1.0 \
    --save_steps 500 \
    --save_total_limit 2

## 7. Monitor Training with TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/checkpoints

## 8. View Secured Checkpoints on Google Drive

In [ ]:
import os
drive_ckpts = "/content/drive/MyDrive/Video-LLaVA/checkpoints"
if os.path.exists(drive_ckpts):
    print("\u2705 Secured Checkpoints on Google Drive:")
    for root, dirs, files in os.walk(drive_ckpts):
        level = root.replace(drive_ckpts, '').count(os.sep)
        indent = ' ' * 4 * level
        print(f"{indent}{os.path.basename(root)}/")
        subindent = ' ' * 4 * (level + 1)
        for f in files:
            if f.endswith('.bin') or f.endswith('.json'):
                print(f"{subindent}{f}")
else:
    print("No checkpoints on Drive yet.")